# DecodeMultiResizeCropEmbed + RandomBackgroundBlend

Demonstrates the fused decode-resize-crop-embed pipeline for variable-size
multi-crop SSL training with background blending.

**Problem:** When crop sizes vary per image (e.g., `size=(64, 224)`), the
batch can't be stacked into a single tensor, making downstream GPU transforms
impossible at batch speed.

**Solution:** `DecodeMultiResizeCropEmbed` embeds each crop onto a fixed-size
zero canvas with a binary alpha channel (RGBA), guaranteeing uniform output
size. `RandomBackgroundBlend` then composites the foreground onto a generated
background using the alpha channel, with optional edge fading.

**Pipeline:**
```
DecodeMultiResizeCropEmbed  →  ToTorchImage  →  RandomBackgroundBlend
   [B, canvas, canvas, 4]     [B, 4, H, W]      [B, 3, H, W]
   numpy HWC RGBA uint8       torch CHW float    torch CHW float (optionally normalized)
```

**Key classes:**
- `DecodeMultiResizeCropEmbed`: resize short edge, crop long edge, embed on RGBA canvas
- `RandomBackgroundBlend`: alpha-composite onto generated background with optional fade

In [ ]:
LITDATA_VAL_PATH = "s3://visionlab-datasets/imagenet1k/pre-processed/s256-l512-jpgbytes-q100-streaming/val/"

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from slipstream import (
    SlipstreamDataset,
    SlipstreamLoader,
    DecodeMultiResizeCropEmbed,
    DecodeMultiRandomResizeShortCropLong,
    MultiCropPipeline,
    ToTorchImage,
    Normalize,
    RandomBackgroundBlend,
    IMAGENET_MEAN,
    IMAGENET_STD,
)

dataset = SlipstreamDataset(
    remote_dir=LITDATA_VAL_PATH,
    decode_images=False,
)
print(f"Dataset: {len(dataset):,} samples")

In [ ]:
def show_embedded(images, n_cols=8, suptitle=None, row_labels=None,
                  mean=None, std=None):
    """Show a batch or list of batches of CHW float tensors.

    If mean/std are provided, images are denormalized before display.
    """
    if isinstance(images, torch.Tensor) and images.ndim == 4:
        images = [images]
    n_rows = len(images)
    n_cols = min(n_cols, images[0].shape[0])
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.5 * n_cols, 2.8 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]
    for r, batch in enumerate(images):
        for c in range(n_cols):
            img = batch[c].clone()
            if mean is not None and std is not None:
                for ch in range(min(3, img.shape[0])):
                    img[ch] = img[ch] * std[ch] + mean[ch]
            img = img[:3].permute(1, 2, 0).clamp(0, 1).cpu().numpy()
            axes[r, c].imshow(img)
            axes[r, c].axis('off')
        if row_labels:
            axes[r, 0].set_ylabel(row_labels[r], fontsize=10, rotation=0,
                                   labelpad=60, va='center')
    if suptitle:
        fig.suptitle(suptitle, fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


def show_rgba(rgba_batch, n_cols=8, suptitle=None, row_labels=None):
    """Show RGBA numpy arrays [B, H, W, 4] with alpha overlay."""
    if isinstance(rgba_batch, np.ndarray) and rgba_batch.ndim == 4:
        rgba_batch = [rgba_batch]
    n_rows = len(rgba_batch)
    n_cols = min(n_cols, rgba_batch[0].shape[0])
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.5 * n_cols, 2.8 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]
    for r, batch in enumerate(rgba_batch):
        for c in range(n_cols):
            # Composite RGBA onto a checkerboard for visualization
            h, w = batch.shape[1], batch.shape[2]
            checker = np.zeros((h, w, 3), dtype=np.uint8) + 200
            checker[::16, :, :] = 220
            checker[:, ::16, :] = 220
            alpha = batch[c, :, :, 3:4].astype(np.float32) / 255.0
            rgb = batch[c, :, :, :3].astype(np.float32)
            composite = (alpha * rgb + (1 - alpha) * checker).astype(np.uint8)
            axes[r, c].imshow(composite)
            axes[r, c].axis('off')
        if row_labels:
            axes[r, 0].set_ylabel(row_labels[r], fontsize=10, rotation=0,
                                   labelpad=60, va='center')
    if suptitle:
        fig.suptitle(suptitle, fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

## 1. Basic Usage: Fixed-Size Crops on Canvas

Decode + resize-short + crop-long + embed onto a 320x320 RGBA canvas.
The alpha channel marks where the image is (255) vs background (0).

In [ ]:
decoder = DecodeMultiResizeCropEmbed(
    {
        'large': dict(size=224, x_range=(0, 1), y_range=(0, 1), seed=42),
        'small': dict(size=112, x_range=(0, 1), y_range=(0, 1), seed=43),
    },
    canvas_size=320,
    embed_x_range=(0, 1),
    embed_y_range=(0, 1),
    embed_seed=99,
)

loader = SlipstreamLoader(
    dataset, batch_size=8, shuffle=False,
    pipelines={'image': [decoder]},
    exclude_fields=['path'], verbose=False,
)

batch = next(iter(loader))
loader.shutdown()

for name in ['large', 'small']:
    arr = batch[name]
    alpha = arr[..., 3]
    print(f"{name}: shape={arr.shape}, dtype={arr.dtype}, "
          f"opaque={int((alpha == 255).sum()):,}, transparent={int((alpha == 0).sum()):,}")

show_rgba(
    [batch['large'], batch['small']],
    row_labels=['large (224)', 'small (112)'],
    suptitle='RGBA output: image on 320x320 canvas (gray = transparent)',
)

## 2. Embed Position Control

`embed_x_range` and `embed_y_range` control where the crop is placed on the canvas.
Defaults to center (0.5). Range (0, 1) enables random placement.

In [ ]:
positions = [
    ('center (0.5)', 0.5, 0.5),
    ('top-left (0)', 0.0, 0.0),
    ('bottom-right (1)', 1.0, 1.0),
    ('random (0-1)', (0, 1), (0, 1)),
]

rows = []
labels = []
for label, xr, yr in positions:
    dec = DecodeMultiResizeCropEmbed(
        {'view': dict(size=160, x_range=(0, 1), y_range=(0, 1), seed=42)},
        canvas_size=320, embed_x_range=xr, embed_y_range=yr, embed_seed=99,
    )
    loader = SlipstreamLoader(
        dataset, batch_size=8, shuffle=False,
        pipelines={'image': [dec]},
        exclude_fields=['path'], verbose=False,
    )
    b = next(iter(loader))
    rows.append(b['view'])
    labels.append(label)
    loader.shutdown()

show_rgba(rows, row_labels=labels,
          suptitle='Embed position: where the crop is placed on the canvas')

## 3. Variable Crop Size (per_batch)

With `size=(min, max)`, the crop size varies randomly. `per_batch` mode
samples one size shared by all images in a batch. All outputs are still
the same canvas size, so the batch is always stackable.

In [ ]:
dec = DecodeMultiResizeCropEmbed(
    {'view': dict(size=(64, 224), x_range=(0, 1), y_range=(0, 1), seed=42)},
    canvas_size=320, embed_x_range=(0, 1), embed_y_range=(0, 1),
    embed_seed=99, size_mode='per_batch',
)

loader = SlipstreamLoader(
    dataset, batch_size=8, shuffle=False,
    pipelines={'image': [dec]},
    exclude_fields=['path'], verbose=False,
)

rows = []
labels = []
it = iter(loader)
for i in range(4):
    b = next(it)
    arr = b['view']
    # Infer crop size from alpha bounding box
    alpha_row = (arr[0, :, :, 3] > 0).any(axis=1)
    crop_h = int(alpha_row.sum())
    rows.append(arr)
    labels.append(f'batch {i}: crop={crop_h}px')
    print(f"Batch {i}: output={arr.shape}, crop_size~{crop_h}px")
loader.shutdown()

show_rgba(rows, row_labels=labels,
          suptitle='Variable crop size (64-224) per_batch on 320x320 canvas')

## 4. RandomBackgroundBlend: Background Types

After `ToTorchImage` converts RGBA to a float tensor, `RandomBackgroundBlend`
uses the alpha channel to composite the foreground onto a generated background.

Three background types: zeros (black), mean (constant fill), power_law (1/f^alpha noise).

In [ ]:
# Decode a batch for reuse
dec = DecodeMultiResizeCropEmbed(
    {'view': dict(size=160, x_range=(0, 1), y_range=(0, 1), seed=42)},
    canvas_size=320, embed_x_range=(0, 1), embed_y_range=(0, 1), embed_seed=99,
)
loader = SlipstreamLoader(
    dataset, batch_size=8, shuffle=False,
    pipelines={'image': [dec]},
    exclude_fields=['path'], verbose=False,
)
raw_batch = next(iter(loader))
loader.shutdown()

to_torch = ToTorchImage(device='cpu')
rgba = to_torch(raw_batch['view'])  # [B, 4, 320, 320] float

bg_configs = [
    ('zeros', dict(background='zeros')),
    ('mean (ImageNet)', dict(background='mean', mean=[0.485, 0.456, 0.406])),
    ('power_law (alpha=0)', dict(background='power_law', alpha_range=0.0, seed=42)),
    ('power_law (alpha=1)', dict(background='power_law', alpha_range=1.0, seed=42)),
    ('power_law (alpha=2)', dict(background='power_law', alpha_range=2.0, seed=42)),
]

rows = []
labels = []
for label, kwargs in bg_configs:
    blend = RandomBackgroundBlend(**kwargs)
    rows.append(blend(rgba))
    labels.append(label)

show_embedded(rows, row_labels=labels,
              suptitle='RandomBackgroundBlend: background types')

## 5. Fade Modes: None vs Circular vs Cosine

`RandomBackgroundBlend` supports three edge fading modes:
- `None`: hard rectangular boundary (default)
- `"circular"`: Gaussian radial fade (same as `RandomEmbed` `fade_radius`)
- `"cosine"`: cosine-tapered Tukey window (smooth rectangular edges)

In [ ]:
fade_configs = [
    ('no fade', dict(fade_mode=None)),
    ('circular (0.35, 0.50)', dict(fade_mode='circular', fade_radius=(0.35, 0.50))),
    ('circular (0.10, 0.50)', dict(fade_mode='circular', fade_radius=(0.10, 0.50))),
    ('cosine inset=0.05', dict(fade_mode='cosine', inset=0.05)),
    ('cosine inset=0.15', dict(fade_mode='cosine', inset=0.15)),
    ('cosine inset=0.30', dict(fade_mode='cosine', inset=0.30)),
]

rows = []
labels = []
for label, kwargs in fade_configs:
    blend = RandomBackgroundBlend(
        background='power_law', alpha_range=1.5, seed=42, **kwargs,
    )
    rows.append(blend(rgba))
    labels.append(label)

show_embedded(rows, row_labels=labels,
              suptitle='Fade modes: hard rect vs circular vs cosine')

## 6. Visualise Fade Masks

Show the actual mask shapes for circular and cosine modes.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

# Circular masks
circular_configs = [
    ('(0.48, 0.50)', (0.48, 0.50)),
    ('(0.35, 0.50)', (0.35, 0.50)),
    ('(0.20, 0.50)', (0.20, 0.50)),
    ('(0.10, 0.50)', (0.10, 0.50)),
]
for ax, (label, fr) in zip(axes[0], circular_configs):
    blend = RandomBackgroundBlend(fade_mode='circular', fade_radius=fr)
    mask = blend._make_circular_fade(160, 160, 'cpu', torch.float32)
    ax.imshow(mask.numpy(), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'circular {label}', fontsize=9)
    ax.axis('off')

# Cosine masks
cosine_configs = [
    ('inset=0.02', 0.02),
    ('inset=0.05', 0.05),
    ('inset=0.15', 0.15),
    ('inset=0.30', 0.30),
]
for ax, (label, inset) in zip(axes[1], cosine_configs):
    blend = RandomBackgroundBlend(fade_mode='cosine', inset=inset)
    mask = blend._make_cosine_fade(160, 160, 'cpu', torch.float32)
    ax.imshow(mask.numpy(), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'cosine {label}', fontsize=9)
    ax.axis('off')

fig.suptitle('Fade masks (160x160 crop)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Stochastic Fade (`p_fade`)

`p_fade` controls the per-image probability of applying fade.
At `p_fade=0.5`, roughly half the images get faded edges and half keep
hard boundaries.

In [ ]:
p_configs = [1.0, 0.5, 0.0]
rows = []
labels = []
for p in p_configs:
    blend = RandomBackgroundBlend(
        background='power_law', alpha_range=1.5,
        fade_mode='cosine', inset=0.10, p_fade=p, seed=42,
    )
    rows.append(blend(rgba))
    labels.append(f'p_fade={p}')

show_embedded(rows, row_labels=labels,
              suptitle='Stochastic fade: per-image probability')

## 8. Full Pipeline: Decode + Embed + Blend + Normalize

The complete SSL-ready pipeline. `RandomBackgroundBlend` handles
normalization internally via `mean`/`std`, so no separate `Normalize`
step is needed.

In [ ]:
DEVICE = 'cpu'  # change to 'cuda' for GPU

loader = SlipstreamLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    seed=42,
    pipelines={'image': [
        DecodeMultiResizeCropEmbed(
            {
                'global_view': dict(size=224, x_range=(0, 1), y_range=(0, 1), seed=42),
                'local_view': dict(size=96, x_range=(0, 1), y_range=(0, 1), seed=43),
            },
            canvas_size=320,
            embed_x_range=(0, 1),
            embed_y_range=(0, 1),
            embed_seed=99,
        ),
        MultiCropPipeline({
            'global_view': [
                ToTorchImage(device=DEVICE),
                RandomBackgroundBlend(
                    background='power_law', alpha_range=1.5,
                    fade_mode='cosine', inset=0.05,
                    mean=IMAGENET_MEAN, std=IMAGENET_STD, seed=44,
                ),
            ],
            'local_view': [
                ToTorchImage(device=DEVICE),
                RandomBackgroundBlend(
                    background='power_law', alpha_range=1.5,
                    fade_mode='cosine', inset=0.05,
                    mean=IMAGENET_MEAN, std=IMAGENET_STD, seed=44,
                ),
            ],
        }),
    ]},
    exclude_fields=['path'],
    verbose=False,
)

batch = next(iter(loader))
loader.shutdown()

print(f"global_view: {batch['global_view'].shape}, dtype={batch['global_view'].dtype}")
print(f"  mean={batch['global_view'].mean():.4f}, std={batch['global_view'].std():.4f}")
print(f"local_view:  {batch['local_view'].shape}, dtype={batch['local_view'].dtype}")
print(f"  mean={batch['local_view'].mean():.4f}, std={batch['local_view'].std():.4f}")
print(f"label: {batch['label'].shape}")

show_embedded(
    [batch['global_view'], batch['local_view']],
    row_labels=['global (224)', 'local (96)'],
    mean=IMAGENET_MEAN, std=IMAGENET_STD,
    suptitle='Full pipeline: decode + embed + blend + normalize (denormalized for display)',
)

## 9. Variable-Size Multi-Scale Pipeline

The key use case: crops with `size=(min, max)` produce variable-sized images
that are all embedded onto the same canvas. This enables fast batched GPU
pipelines with per-image size variation.

In [ ]:
loader = SlipstreamLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    seed=42,
    pipelines={'image': [
        DecodeMultiResizeCropEmbed(
            {
                'view_large': dict(size=(112, 224), x_range=(0, 1), y_range=(0, 1), seed=42),
                'view_small': dict(size=(32, 112), x_range=(0, 1), y_range=(0, 1), seed=43),
            },
            canvas_size=320,
            embed_x_range=(0, 1),
            embed_y_range=(0, 1),
            embed_seed=99,
            size_mode='per_batch',
        ),
        MultiCropPipeline({
            'view_large': [
                ToTorchImage(device=DEVICE),
                RandomBackgroundBlend(
                    background='power_law', alpha_range=1.5,
                    fade_mode='cosine', inset=0.08,
                    mean=IMAGENET_MEAN, std=IMAGENET_STD, seed=44,
                ),
            ],
            'view_small': [
                ToTorchImage(device=DEVICE),
                RandomBackgroundBlend(
                    background='power_law', alpha_range=1.5,
                    fade_mode='cosine', inset=0.08,
                    mean=IMAGENET_MEAN, std=IMAGENET_STD, seed=44,
                ),
            ],
        }),
    ]},
    exclude_fields=['path'],
    verbose=False,
)

rows = []
labels = []
it = iter(loader)
for i in range(3):
    b = next(it)
    rows.append(b['view_large'])
    labels.append(f'batch {i} large')
    rows.append(b['view_small'])
    labels.append(f'batch {i} small')
    print(f"Batch {i}: view_large={b['view_large'].shape}, view_small={b['view_small'].shape}")
loader.shutdown()

show_embedded(rows, row_labels=labels,
              mean=IMAGENET_MEAN, std=IMAGENET_STD,
              suptitle='Variable-size multi-scale: different crop sizes across batches')

## 10. Comparison: With vs Without Embed

Side-by-side comparison of `DecodeMultiRandomResizeShortCropLong` (raw crops)
versus `DecodeMultiResizeCropEmbed` (crops on RGBA canvas).

In [ ]:
crop_params = dict(size=160, x_range=(0, 1), y_range=(0, 1), seed=42)

# Without embed
loader_raw = SlipstreamLoader(
    dataset, batch_size=8, shuffle=False,
    pipelines={'image': [
        DecodeMultiRandomResizeShortCropLong({'view': crop_params}),
    ]},
    exclude_fields=['path'], verbose=False,
)

# With embed
loader_embed = SlipstreamLoader(
    dataset, batch_size=8, shuffle=False,
    pipelines={'image': [
        DecodeMultiResizeCropEmbed(
            {'view': crop_params},
            canvas_size=320, embed_x_range=(0, 1), embed_y_range=(0, 1), embed_seed=99,
        ),
    ]},
    exclude_fields=['path'], verbose=False,
)

batch_raw = next(iter(loader_raw))
batch_embed = next(iter(loader_embed))
loader_raw.shutdown()
loader_embed.shutdown()

print(f"Raw crop:  {batch_raw['view'].shape}")
print(f"Embedded:  {batch_embed['view'].shape}")

# Display raw crops as RGB and embedded as RGBA
fig, axes = plt.subplots(2, 8, figsize=(20, 6),
                         gridspec_kw={'height_ratios': [160, 320]})
for c in range(8):
    axes[0, c].imshow(batch_raw['view'][c])
    axes[0, c].axis('off')
    # Embed: composite RGBA
    arr = batch_embed['view'][c]
    alpha = arr[:, :, 3:4].astype(np.float32) / 255.0
    rgb = arr[:, :, :3].astype(np.float32)
    checker = np.full_like(rgb, 200)
    composite = (alpha * rgb + (1 - alpha) * checker).astype(np.uint8)
    axes[1, c].imshow(composite)
    axes[1, c].axis('off')
axes[0, 0].set_ylabel('raw crop\n160x160', fontsize=10, rotation=0, labelpad=55, va='center')
axes[1, 0].set_ylabel('embedded\n320x320 RGBA', fontsize=10, rotation=0, labelpad=55, va='center')
fig.suptitle('Raw crop vs RGBA-embedded (gray = transparent)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Inspecting Embed Rects

The decoder stores `embed_rects` in `last_params` — the (x, y, w, h)
pixel coordinates of where each crop was placed on the canvas.

In [ ]:
dec = DecodeMultiResizeCropEmbed(
    {
        'large': dict(size=224, x_range=(0, 1), y_range=(0, 1), seed=42),
        'small': dict(size=96, x_range=(0, 1), y_range=(0, 1), seed=43),
    },
    canvas_size=320,
    embed_x_range=(0, 1), embed_y_range=(0, 1), embed_seed=99,
)

loader = SlipstreamLoader(
    dataset, batch_size=4, shuffle=False,
    pipelines={'image': [dec]},
    exclude_fields=['path'], verbose=False,
)
batch = next(iter(loader))
loader.shutdown()

params = dec.last_params
print("Embed rects (x, y, w, h):")
for name in ['large', 'small']:
    print(f"  {name}:")
    for i in range(4):
        r = params['embed_rects'][name][i]
        print(f"    image {i}: x={r[0]}, y={r[1]}, w={r[2]}, h={r[3]}")

# Visualize rects overlaid on canvas
fig, axes = plt.subplots(1, 4, figsize=(12, 3.5))
colors = {'large': 'red', 'small': 'blue'}
for i, ax in enumerate(axes):
    ax.imshow(batch['large'][i, :, :, :3])
    for name, color in colors.items():
        r = params['embed_rects'][name][i]
        rect = patches.Rectangle((r[0], r[1]), r[2], r[3],
                                  linewidth=2, edgecolor=color,
                                  facecolor='none', linestyle='--')
        ax.add_patch(rect)
    ax.axis('off')
    ax.set_title(f'image {i}', fontsize=10)

# Legend
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], color=c, lw=2, linestyle='--', label=n)
                   for n, c in colors.items()]
fig.legend(handles=legend_elements, loc='lower center', ncol=2, fontsize=10)
fig.suptitle('Embed rects: where each crop was placed on the canvas',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 12. Color vs Grayscale Noise

`color_noise=True` (default) generates independent noise per R, G, B channel.
`color_noise=False` duplicates a single noise field across channels (grayscale).

In [ ]:
rows = []
labels = []
for cn, label in [(True, 'color (default)'), (False, 'grayscale')]:
    blend = RandomBackgroundBlend(
        background='power_law', alpha_range=1.5, color_noise=cn,
        fade_mode='cosine', inset=0.05, seed=42,
    )
    rows.append(blend(rgba))
    labels.append(label)

show_embedded(rows, row_labels=labels,
              suptitle='Color noise vs grayscale noise backgrounds')

## Summary

### DecodeMultiResizeCropEmbed

| Parameter | Effect |
|-----------|--------|
| `size=224` | Fixed crop size |
| `size=(64, 224)` | Variable crop size (per_batch or per_image) |
| `canvas_size=320` | Output canvas size (always square, RGBA) |
| `embed_x_range=(0, 1)` | Random horizontal placement |
| `embed_y_range=0.5` | Vertically centered |
| `embed_seed=99` | Reproducible placement |

### RandomBackgroundBlend

| Parameter | Effect |
|-----------|--------|
| `background="zeros"` | Black background |
| `background="mean"` | Constant fill |
| `background="power_law"` | 1/f^alpha colored noise |
| `alpha_range=1.5` | Noise exponent (0=white, 2=Brownian) |
| `color_noise=True` | Independent per-channel noise |
| `fade_mode=None` | Hard rectangular boundary |
| `fade_mode="circular"` | Gaussian radial fade |
| `fade_mode="cosine"` | Cosine-tapered Tukey window |
| `fade_radius=(0.35, 0.50)` | Circular fade inner/outer |
| `inset=0.05` | Cosine fade width (fraction of crop dim) |
| `p_fade=0.5` | Per-image fade probability |
| `mean=..., std=...` | Normalize output (no separate Normalize needed) |